In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\Aniket\AppData\Local\Temp\ipykernel_42716\3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [27]:
from langchain_ollama import OllamaEmbeddings , ChatOllama

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024
)

llm = ChatOllama(
    model="gemma3:4b" 
)

In [29]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [3]:
video_id = 'UabBYexBD4k'
ytt_api = YouTubeTranscriptApi()

transcript_list = ytt_api.fetch(video_id , languages=["en", "hi"])

In [4]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="There's a fundamental truth about LLMs, large\xa0\nlanguage models. They are frozen in time. They\xa0\xa0", start=0.32, duration=9.44), FetchedTranscriptSnippet(text='know everything about our world up until\xa0\ntheir training cutoff date and absolutely\xa0\xa0', start=9.76, duration=6.24), FetchedTranscriptSnippet(text='nothing about what happened 5 minutes ago. Nor\xa0\ndo they know anything about your private data,\xa0\xa0', start=16.0, duration=5.68), FetchedTranscriptSnippet(text='your internal wikis, your proprietary codebase. And\xa0\nif we do want an LLM to know any of that stuff,\xa0\xa0', start=21.68, duration=6.56), FetchedTranscriptSnippet(text='well, we have to solve the problem of context\xa0\ninjection. How do we get the right data into the\xa0\xa0', start=28.24, duration=7.76), FetchedTranscriptSnippet(text='model at the right time? And there have been two\xa0\nvery different ways to handle this. Now, the first

In [5]:
print(type(transcript_list))
print(type(transcript_list.snippets[0]))

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>
<class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>


In [6]:
from langchain_core.documents import Document

transcript_text = " ".join(
    snippet.text
    for snippet in transcript_list
)

doc = Document(
    page_content=transcript_text,
    metadata={"source": video_id}
)

In [7]:
doc.page_content = doc.page_content.replace("\xa0\n", " ").replace("\xa0", " ")

In [8]:
doc

Document(metadata={'source': 'UabBYexBD4k'}, page_content="There's a fundamental truth about LLMs, large language models. They are frozen in time. They   know everything about our world up until their training cutoff date and absolutely   nothing about what happened 5 minutes ago. Nor do they know anything about your private data,   your internal wikis, your proprietary codebase. And if we do want an LLM to know any of that stuff,   well, we have to solve the problem of context injection. How do we get the right data into the   model at the right time? And there have been two very different ways to handle this. Now, the first   is really what we can think of as the engineering approach. It's RAG, retrieval augmented generation.   So here we've got an LLM and we've also got an input prompt from the user. Now ahead of time   we take some of the documents that we want to give to this LLM. So these are documents that could be   PDFs or code files or entire books and we chunk them. We break

In [9]:
doc.metadata['source'] = f"https://www.youtube.com/watch?v={doc.metadata['source']}"

In [10]:
doc

Document(metadata={'source': 'https://www.youtube.com/watch?v=UabBYexBD4k'}, page_content="There's a fundamental truth about LLMs, large language models. They are frozen in time. They   know everything about our world up until their training cutoff date and absolutely   nothing about what happened 5 minutes ago. Nor do they know anything about your private data,   your internal wikis, your proprietary codebase. And if we do want an LLM to know any of that stuff,   well, we have to solve the problem of context injection. How do we get the right data into the   model at the right time? And there have been two very different ways to handle this. Now, the first   is really what we can think of as the engineering approach. It's RAG, retrieval augmented generation.   So here we've got an LLM and we've also got an input prompt from the user. Now ahead of time   we take some of the documents that we want to give to this LLM. So these are documents that could be   PDFs or code files or entire b

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter( 
    chunk_size=1000,
    chunk_overlap=150, 
)

In [12]:
doc = text_splitter.split_documents([doc])

In [13]:
doc

[Document(metadata={'source': 'https://www.youtube.com/watch?v=UabBYexBD4k'}, page_content="There's a fundamental truth about LLMs, large language models. They are frozen in time. They   know everything about our world up until their training cutoff date and absolutely   nothing about what happened 5 minutes ago. Nor do they know anything about your private data,   your internal wikis, your proprietary codebase. And if we do want an LLM to know any of that stuff,   well, we have to solve the problem of context injection. How do we get the right data into the   model at the right time? And there have been two very different ways to handle this. Now, the first   is really what we can think of as the engineering approach. It's RAG, retrieval augmented generation.   So here we've got an LLM and we've also got an input prompt from the user. Now ahead of time   we take some of the documents that we want to give to this LLM. So these are documents that could be   PDFs or code files or entire 

In [14]:
from langchain_chroma import Chroma

In [15]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="video_db",
    collection_name="Youtube_Video_Collection"
)

In [16]:
vector_store.add_documents(doc)

['8e22723a-83a3-4286-bedf-5555c19f0955',
 '51598182-ff21-4acc-a9be-fb5d7bbafb4d',
 'c31000b9-017a-4c2f-982d-42f9f077d693',
 'cb8e52ad-762a-4bcc-a818-21be5fa8c5c1',
 '0b434d98-e7df-492c-bd0d-cb37899e191e',
 '0d154c1e-5b7c-4f45-956c-c02c71a0c25c',
 '2f0d2a21-1eb8-4049-a661-57e6c02ab005',
 '467b033f-303d-4e44-8f4d-e356a564bb19',
 '350c1898-ced4-419e-907c-cac02b4bde55',
 '13e1c734-3c93-41aa-ae5a-2287b87c6038',
 '59844cf7-918d-4545-bbd9-8a2fc6d7cbbe']

In [19]:
base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [30]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=model
)

In [21]:
questions = [
    "What is the fundamental truth about LLMs mentioned at the beginning?",
    "What are the two approaches discussed for injecting external knowledge into an LLM?",
    "In a RAG pipeline, what role does the embedding model play?",
    "What is stored inside the vector database?",
    "Why were long-context approaches not practical in early LLMs?",
    "The speaker gives an example of how many tokens some modern models can handle. What was the approximate number?",
    "What famous books were used to illustrate the size of a million-token context window?",
    "According to the speaker, what are the major infrastructure components of a production RAG system?",
    "What does the speaker call the problem where relevant information exists but is not retrieved?",
    "Why does the speaker describe semantic search as probabilistic?",
    "Explain why the 'omitted security requirements' example is difficult for traditional RAG.",
    "Why can long-context models sometimes perform better on comparison tasks across multiple documents?",
    "What tradeoff exists between simplicity and efficiency when comparing Long Context and RAG?",
    "Why might a model fail to answer correctly even when the answer is present somewhere in a 500k-token context window?",
    "How does RAG attempt to solve the 'needle in the haystack' problem?",
    "Suppose a company has 20 TB of internal documents. Why would long context alone be insufficient according to the speaker?",
    "If prompt caching exists, why doesn't it completely eliminate long-context costs?",
    "The speaker says RAG pays a processing cost 'once at indexing time.' Explain what this means.",
    "If a retrieval system returns the wrong chunks, can the LLM still find the correct answer? Why or why not?",
    "Why does the speaker argue that vector databases are still relevant despite million-token context windows?",
    "A user asks: 'Which security requirements were removed before release?' What retrieval strategy would likely work better (Standard similarity search, Long context, Multi-query retrieval, or Contextual retrieval) and why?",
    "Give one scenario where Long Context is clearly superior to RAG.",
    "Give one scenario where RAG is clearly superior to Long Context.",
    "If you had to design an enterprise chatbot today, would you choose Pure RAG, Pure Long Context, or Hybrid? Defend your answer using arguments from the transcript.",
    "The transcript's central question is: 'Is RAG becoming an unnecessary complexity layer?' Give the strongest argument for that claim and the strongest argument against it."
]

In [ ]:
len(questions)

In [31]:
for question in questions[:5]:
    print(f"Question: {question}")
    retrieved_docs = retriever.invoke(question)
    for i, doc in enumerate(retrieved_docs):
        print(f"Retrieved Document {i+1}: {doc.metadata['source']} - {doc.page_content}...")
    print("\n---\n")

Question: What is the fundamental truth about LLMs mentioned at the beginning?
Retrieved Document 1: https://www.youtube.com/watch?v=UabBYexBD4k - There's a fundamental truth about LLMs, large language models. They are frozen in time. They   know everything about our world up until their training cutoff date and absolutely   nothing about what happened 5 minutes ago. Nor do they know anything about your private data,   your internal wikis, your proprietary codebase. And if we do want an LLM to know any of that stuff,   well, we have to solve the problem of context injection. How do we get the right data into the   model at the right time? And there have been two very different ways to handle this. Now, the first   is really what we can think of as the engineering approach. It's RAG, retrieval augmented generation.   So here we've got an LLM and we've also got an input prompt from the user. Now ahead of time   we take some of the documents that we want to give to this LLM. So these are 

In [32]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [33]:
question          = "Suppose a company has 20 TB of internal documents. Why would long context alone be insufficient according to the speaker?"
retrieved_docs    = retriever.invoke(question)

In [34]:
retrieved_docs

[Document(id='59844cf7-918d-4545-bbd9-8a2fc6d7cbbe', metadata={'source': 'https://www.youtube.com/watch?v=UabBYexBD4k'}, page_content="tokens sounds great but in   the scheme of enterprise data that's really just a drop in the bucket. I mean an enterprise data lake   that's probably measured in terabytes or or maybe even petabytes. So if you want an infinite data   set that stores everything, you really do need to have a retrieval layer to filter information down   to something that fits into the LLM context window. So where does this leave us? Well,   if your problem involves a bounded data set and requires complex global reasoning like analyzing   a specific legal contract or summarizing a book, I think long context is the way to go. It simplifies   the stack and it improves the reasoning. But if you're navigating the infinite data set of   enterprise knowledge, the vector database remains the only viable warehouse for your data. But how   about you? Are you team long context, team R

In [35]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"tokens sounds great but in   the scheme of enterprise data that's really just a drop in the bucket. I mean an enterprise data lake   that's probably measured in terabytes or or maybe even petabytes. So if you want an infinite data   set that stores everything, you really do need to have a retrieval layer to filter information down   to something that fits into the LLM context window. So where does this leave us? Well,   if your problem involves a bounded data set and requires complex global reasoning like analyzing   a specific legal contract or summarizing a book, I think long context is the way to go. It simplifies   the stack and it improves the reasoning. But if you're navigating the infinite data set of   enterprise knowledge, the vector database remains the only viable warehouse for your data. But how   about you? Are you team long context, team RAG, maybe a bit of both? Let me know in the comments.\n\nbrute force approach and that one is called   long context. Now this is reall

In [36]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [37]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      tokens sounds great but in   the scheme of enterprise data that's really just a drop in the bucket. I mean an enterprise data lake   that's probably measured in terabytes or or maybe even petabytes. So if you want an infinite data   set that stores everything, you really do need to have a retrieval layer to filter information down   to something that fits into the LLM context window. So where does this leave us? Well,   if your problem involves a bounded data set and requires complex global reasoning like analyzing   a specific legal contract or summarizing a book, I think long context is the way to go. It simplifies   the stack and it improves the reasoning. But if you're navigating the infinite data set of   enterprise knowledge, the vector database remains the only viable warehouse for your data

In [38]:
answer = llm.invoke(final_prompt)
print(answer.content)

According to the speaker, long context alone would be insufficient because an enterprise data lake is probably measured in terabytes or even petabytes—a drop in the bucket compared to the vast amount of data.


In [ ]:
answer = model.invoke(final_prompt)
print(answer.content[0])

[{'type': 'text', 'text': 'According to the speaker, long context alone is insufficient for 20 TB of data because a context window of millions of tokens is "just a drop in the bucket" compared to enterprise data, which is measured in terabytes or petabytes. For an infinite data set of enterprise knowledge, a retrieval layer is necessary to filter information down to fit into the LLM context window.', 'extras': {'signature': 'EjQKMgEMOdbHDCTVeIqh1ve9vP76zh2XyvywYNPGoTKZRGNkWD6L6gI0SCFmkrqz+tYI/HDy'}}]


In [41]:
print(answer.content[0]['text'])

According to the speaker, long context alone is insufficient for 20 TB of data because a context window of millions of tokens is "just a drop in the bucket" compared to enterprise data, which is measured in terabytes or petabytes. For an infinite data set of enterprise knowledge, a retrieval layer is necessary to filter information down to fit into the LLM context window.
